In [ ]:
!pip -q install --no-cache-dir "numpy==1.26.4" "pandas==2.2.2"
!pip -q install --no-cache-dir "opencv-python-headless==4.10.0.84"
!pip -q install --no-cache-dir decord tqdm einops
!pip -q install --no-cache-dir torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install --no-cache-dir "ultralytics==8.3.0"
!pip -q install --no-cache-dir "emotiefflib[torch]"


In [ ]:
import os
from pathlib import Path

ROOT = "/content/miga_data"
SRC_DIR = "/content/src"

Path(ROOT).mkdir(parents=True, exist_ok=True)
Path(SRC_DIR).mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("SRC_DIR:", SRC_DIR)


ROOT: /content/miga_data
SRC_DIR: /content/src


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


FEAT_DIR = "/content/drive/MyDrive/miga_features_cache_agcn"


Mounted at /content/drive


In [ ]:
DATA_DIR = "/content/miga_data"
!mkdir -p $DATA_DIR

!wget -O $DATA_DIR/imigue_skeleton_phase1.zip https://miga3.a3s.fi/imigue_skeleton_phase1.zip

!wget -O $DATA_DIR/imigue_rgb_phase1.zip https://miga3.a3s.fi/imigue_rgb_phase1.zip

!wget -O $DATA_DIR/imigue_skeleton_phase2.zip https://miga3.a3s.fi/imigue_skeleton_phase2.zip

!wget -O $DATA_DIR/imigue_rgb_phase2.zip https://miga3.a3s.fi/imigue_rgb_phase2.zip

--2026-01-17 13:19:13--  https://miga3.a3s.fi/imigue_skeleton_phase1.zip
Resolving miga3.a3s.fi (miga3.a3s.fi)... 86.50.254.18, 86.50.254.19
Connecting to miga3.a3s.fi (miga3.a3s.fi)|86.50.254.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6270753807 (5.8G) [application/zip]
Saving to: ‘/content/miga_data/imigue_skeleton_phase1.zip’

/content/miga_data/ 100%[===================>]   5.84G  19.2MB/s    in 5m 17s  

2026-01-17 13:24:32 (18.8 MB/s) - ‘/content/miga_data/imigue_skeleton_phase1.zip’ saved [6270753807/6270753807]

--2026-01-17 13:24:32--  https://miga3.a3s.fi/imigue_rgb_phase1.zip
Resolving miga3.a3s.fi (miga3.a3s.fi)... 86.50.254.18, 86.50.254.19
Connecting to miga3.a3s.fi (miga3.a3s.fi)|86.50.254.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6512356945 (6.1G) [application/zip]
Saving to: ‘/content/miga_data/imigue_rgb_phase1.zip’

/content/miga_data/ 100%[===================>]   6.06G  20.5MB/s    in 5m 19s  

202

In [ ]:
!unzip -q $DATA_DIR/imigue_skeleton_phase1.zip -d $DATA_DIR
!unzip -q $DATA_DIR/imigue_rgb_phase1.zip -d $DATA_DIR
!unzip -q $DATA_DIR/imigue_skeleton_phase2.zip -d $DATA_DIR
!unzip -q $DATA_DIR/imigue_rgb_phase2.zip -d $DATA_DIR

In [ ]:
!rm $DATA_DIR/*.zip

In [ ]:
%%writefile /content/src/__init__.py

Overwriting /content/src/__init__.py


In [ ]:
%%writefile /content/src/paths.py
import os, glob

ROOT = "/content/miga_data"

RGB_P1_TRAIN = f"{ROOT}/imigue_rgb_phase1/train_data"
RGB_P1_VAL   = f"{ROOT}/imigue_rgb_phase1/validation_data"
RGB_P2       = f"{ROOT}/imigue_rgb_phase2"

SK_P1_TRAIN  = f"{ROOT}/imigue_data_phase1/datasets/imigue_skeleton_train"
SK_P1_VAL    = f"{ROOT}/imigue_data_phase1/datasets/imigue_skeleton_validate"
SK_P2_TEST   = f"{ROOT}/imigue_data_phase2/imigue_skeleton_test"

def vid4(x): return f"{int(x):04d}"

def resolve_video_path_phase1(video_id, split):
    v = vid4(video_id)
    if split == "train":
        p = os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    if split == "val":
        p = os.path.join(RGB_P1_VAL, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    for p in [
        os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4"),
        os.path.join(RGB_P1_VAL, v, f"{v}.mp4"),
    ]:
        if os.path.exists(p): return p
    return None

def resolve_video_path_phase2(video_id):
    v = vid4(video_id)
    p = os.path.join(RGB_P2, v, f"{v}.mp4")
    if os.path.exists(p): return p
    hits = glob.glob(os.path.join(RGB_P2, "**", f"{v}.mp4"), recursive=True)
    return hits[0] if hits else None

def resolve_skeleton_path_phase1(video_id, split, prefer_hand=True):
    v = vid4(video_id)
    base = SK_P1_TRAIN if split=="train" else SK_P1_VAL
    p_hand  = os.path.join(base, v, f"{v}_light_hand.csv")
    p_light = os.path.join(base, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None

def resolve_skeleton_path_phase2(video_id, prefer_hand=True):
    v = vid4(video_id)
    p_hand  = os.path.join(SK_P2_TEST, v, f"{v}_light_hand.csv")
    p_light = os.path.join(SK_P2_TEST, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None

Overwriting /content/src/paths.py


In [ ]:
import os
import pandas as pd
import glob, os

ROOT = "/content/miga_data"

train_csv = f"{ROOT}/imigue_rgb_phase1/train_label.csv"
val_csv   = f"{ROOT}/imigue_rgb_phase1/validation_label.csv"

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)

train_df["split"] = "train"
val_df["split"] = "val"

phase1_all = pd.concat([train_df, val_df], ignore_index=True)

def vid4(x):
    return f"{int(x):04d}"

def video_path(row):
    v = vid4(row["video_id"])
    if row["split"] == "train":
        return f"{ROOT}/imigue_rgb_phase1/train_data/{v}/{v}.mp4"
    else:
        return f"{ROOT}/imigue_rgb_phase1/validation_data/{v}/{v}.mp4"

def skeleton_path(row):
    v = f"{int(row['video_id']):04d}"
    roots = [
        "/content/miga_data/imigue_data_phase1",
        "/content/miga_data",
    ]
    for root in roots:
        hits = glob.glob(f"{root}/**/{v}*_light*.csv", recursive=True)
        if hits:
            return hits[0]
    return None


phase1_all["skeleton_path"] = phase1_all.apply(skeleton_path, axis=1)
phase1_all.to_csv("/content/phase1_all_with_split.csv", index=False)


In [ ]:
%%writefile /content/src/agcn.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class AGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_t = nn.Conv2d(in_channels, out_channels, kernel_size=(9, 1), padding=(4, 0))
        self.conv_v = nn.Conv2d(out_channels, out_channels, kernel_size=(1, 1))
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv_t(x)
        x = self.conv_v(x)
        x = self.bn(x)
        return F.relu(x)


class AGCNModel(nn.Module):

    def __init__(self, in_channels=3, hidden=64, out_dim=512):
        super().__init__()

        self.data_bn = nn.BatchNorm1d(in_channels * 25)

        self.block1 = AGCNBlock(in_channels, hidden)
        self.block2 = AGCNBlock(hidden, hidden * 2)
        self.block3 = AGCNBlock(hidden * 2, hidden * 4)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(hidden * 4, out_dim)

    def forward(self, x):
        B, T, V, C = x.shape

        x = x.permute(0, 3, 1, 2).contiguous()

        x = x.view(B, C * V, T)
        x = self.data_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.pool(x).view(B, -1)
        x = self.fc(x)
        return x


Overwriting /content/src/agcn.py


In [ ]:
%%writefile /content/src/feature_extract.py
import os
import glob
import hashlib
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from decord import VideoReader, cpu


MIGA_ROOT = os.environ.get("MIGA_ROOT", "/content/miga_data")

RGB_P1_TRAIN = f"{MIGA_ROOT}/imigue_rgb_phase1/train_data"
RGB_P1_VAL   = f"{MIGA_ROOT}/imigue_rgb_phase1/validation_data"
RGB_P2       = f"{MIGA_ROOT}/imigue_rgb_phase2"

SK_P1_TRAIN  = f"{MIGA_ROOT}/imigue_data_phase1/datasets/imigue_skeleton_train"
SK_P1_VAL    = f"{MIGA_ROOT}/imigue_data_phase1/datasets/imigue_skeleton_validate"
SK_P2_TEST   = f"{MIGA_ROOT}/imigue_data_phase2/imigue_skeleton_test"


def vid4(x: int) -> str:
    return f"{int(x):04d}"


def resolve_video_path_phase1(video_id: int, split: str) -> Optional[str]:
    v = vid4(video_id)
    if split == "train":
        p = os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    if split == "val":
        p = os.path.join(RGB_P1_VAL, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    for p in [
        os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4"),
        os.path.join(RGB_P1_VAL, v, f"{v}.mp4"),
    ]:
        if os.path.exists(p):
            return p
    return None


def resolve_video_path_phase2(video_id: int) -> Optional[str]:
    v = vid4(video_id)
    p = os.path.join(RGB_P2, v, f"{v}.mp4")
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(RGB_P2, "**", f"{v}.mp4"), recursive=True)
    return hits[0] if hits else None


def resolve_skeleton_path_phase1(video_id: int, split: str, prefer_hand: bool = True) -> Optional[str]:
    v = vid4(video_id)
    base = SK_P1_TRAIN if split == "train" else SK_P1_VAL
    p_hand  = os.path.join(base, v, f"{v}_light_hand.csv")
    p_light = os.path.join(base, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None


def resolve_skeleton_path_phase2(video_id: int, prefer_hand: bool = True) -> Optional[str]:
    v = vid4(video_id)
    p_hand  = os.path.join(SK_P2_TEST, v, f"{v}_light_hand.csv")
    p_light = os.path.join(SK_P2_TEST, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None



def cache_key(video_id: int, split: str, phase: int) -> str:
    s = f"{int(video_id)}|{split}|{phase}"
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]


def cache_paths(feat_dir: str, video_id: int, split: str, phase: int) -> Dict[str, str]:
    key = cache_key(video_id, split, phase)
    d = Path(feat_dir)
    d.mkdir(parents=True, exist_ok=True)
    return {
        "ctx":   str(d / f"{key}_ctx.pt"),
        "face":  str(d / f"{key}_face.pt"),
        "fmask": str(d / f"{key}_facemask.pt"),
        "skel":  str(d / f"{key}_skel.pt"),
        "smask": str(d / f"{key}_skelmask.pt"),
    }



def _vr(path: str) -> VideoReader:
    return VideoReader(path, ctx=cpu(0))



_CTX_MODEL = None

def build_ctx_model(device: str = "cuda"):
    global _CTX_MODEL
    if _CTX_MODEL is not None:
        _CTX_MODEL.to(device)
        _CTX_MODEL.eval()
        return _CTX_MODEL

    import torchvision
    from torchvision.models.video import r3d_18, R3D_18_Weights

    weights = R3D_18_Weights.DEFAULT
    model = r3d_18(weights=weights)
    model.fc = nn.Identity()
    model = model.to(device).eval()
    _CTX_MODEL = model
    return _CTX_MODEL


def _preprocess_clip_torch(clip_rgb: np.ndarray, device: str) -> torch.Tensor:
    x = torch.from_numpy(clip_rgb).to(torch.float32) / 255.0
    x = x.permute(3, 0, 1, 2)
    x = x.unsqueeze(0)

    x = F.interpolate(x, size=(x.shape[2], 112, 112), mode="trilinear", align_corners=False)

    mean = torch.tensor([0.43216, 0.394666, 0.37645], device=x.device).view(1,3,1,1,1)
    std  = torch.tensor([0.22803, 0.22145, 0.216989], device=x.device).view(1,3,1,1,1)
    x = (x - mean) / std
    return x.to(device)


@torch.no_grad()
def extract_ctx_stream(video_path: str, device: str = "cuda", chunk: int = 32) -> torch.Tensor:
    model = build_ctx_model(device=device)
    vr = _vr(video_path)
    T = len(vr)
    if T <= 0:
        return torch.zeros((0, 512), dtype=torch.float32)

    pad = (-T) % chunk
    total = T + pad

    feats = []
    for i in range(0, total, chunk):
        idxs = list(range(i, min(i + chunk, T)))
        if len(idxs) == 0:
            break
        frames = vr.get_batch(idxs).asnumpy()
        if frames.ndim != 4 or frames.shape[-1] != 3:
            raise RuntimeError(f"Bad frames shape from decord: {frames.shape}")

        if frames.shape[0] < chunk:
            frames = np.concatenate([frames, np.repeat(frames[-1:], chunk - frames.shape[0], axis=0)], axis=0)

        x = _preprocess_clip_torch(frames, device=device)
        f = model(x).squeeze(0).detach().cpu()
        feats.append(f)

    return torch.stack(feats, dim=0).to(torch.float32)




_FACE_MODEL = None
_EMO_REC = None

def _init_yolo():
    global _FACE_MODEL
    if _FACE_MODEL is not None:
        return _FACE_MODEL
    try:
        from ultralytics import YOLO
        for w in ["yolov8n-face.pt", "yolov8n.pt"]:
            try:
                _FACE_MODEL = YOLO(w)
                break
            except Exception:
                _FACE_MODEL = None
    except Exception:
        _FACE_MODEL = None
    return _FACE_MODEL


def _center_crop(img, size=224):
    h, w = img.shape[:2]
    s = min(h, w)
    y0 = (h - s) // 2
    x0 = (w - s) // 2
    crop = img[y0:y0+s, x0:x0+s]
    import cv2
    crop = cv2.resize(crop, (size, size))
    return crop


def _detect_face_crop(img_rgb, size=224):
    model = _init_yolo()
    if model is None:
        return _center_crop(img_rgb, size=size), False

    try:
        res = model.predict(source=img_rgb, verbose=False)
        if len(res) == 0 or res[0].boxes is None or len(res[0].boxes) == 0:
            return _center_crop(img_rgb, size=size), False
        boxes = res[0].boxes.xyxy.detach().cpu().numpy()
        areas = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        b = boxes[int(np.argmax(areas))]
        x1, y1, x2, y2 = [int(max(0, v)) for v in b]
        x2 = min(x2, img_rgb.shape[1]-1)
        y2 = min(y2, img_rgb.shape[0]-1)
        crop = img_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            return _center_crop(img_rgb, size=size), False
        import cv2
        crop = cv2.resize(crop, (size, size))
        return crop, True
    except Exception:
        return _center_crop(img_rgb, size=size), False


def _init_emotieff(device: str):
    global _EMO_REC
    if _EMO_REC is not None:
        return _EMO_REC

    from emotiefflib.facial_analysis import EmotiEffLibRecognizerTorch

    try:
        _EMO_REC = EmotiEffLibRecognizerTorch(device=device)
    except TypeError:
        _EMO_REC = EmotiEffLibRecognizerTorch()
    return _EMO_REC


def _as_rgb_uint8(face_rgb_224):
    x = face_rgb_224
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    if x.dtype != np.uint8:
        x = np.clip(x, 0, 255).astype(np.uint8)
    return x


@torch.no_grad()
def emotieff_embed(face_rgb_224: np.ndarray, device: str = "cuda") -> np.ndarray:

    rec = _init_emotieff(device=device)
    x = _as_rgb_uint8(face_rgb_224)

    for name in [
        "get_face_embedding",
        "get_embedding",
        "get_embeddings",
        "extract_embedding",
        "extract_embeddings",
        "get_features",
        "extract_features",
    ]:
        if hasattr(rec, name) and callable(getattr(rec, name)):
            out = getattr(rec, name)(x)
            out = np.asarray(out).reshape(-1).astype(np.float32)
            return out

    if callable(rec):
        out = rec(x)
        if isinstance(out, dict):
            for k in ["embedding", "emb", "features", "feature"]:
                if k in out:
                    return np.asarray(out[k]).reshape(-1).astype(np.float32)
        out = np.asarray(out).reshape(-1).astype(np.float32)
        return out

    for attr in ["model", "net", "backbone"]:
        if hasattr(rec, attr):
            m = getattr(rec, attr)
            if isinstance(m, nn.Module):
                m = m.to(device).eval()
                t = torch.from_numpy(x).to(torch.float32) / 255.0
                t = t.permute(2, 0, 1).unsqueeze(0).to(device)

                for nm in ["forward_features", "extract_features", "features"]:
                    if hasattr(m, nm) and callable(getattr(m, nm)):
                        feat = getattr(m, nm)(t)
                        feat = feat.reshape(feat.shape[0], -1)
                        return feat.squeeze(0).detach().cpu().numpy().astype(np.float32)

                feat = m(t)
                feat = feat.reshape(feat.shape[0], -1)
                return feat.squeeze(0).detach().cpu().numpy().astype(np.float32)

    raise RuntimeError("EmotiEffLib: не удалось получить embedding")


@torch.no_grad()
def extract_face_windows(video_path: str, device: str = "cuda", chunk: int = 32) -> Tuple[torch.Tensor, torch.Tensor]:

    vr = _vr(video_path)
    T = len(vr)
    if T <= 0:
        return torch.zeros((0, 1280), dtype=torch.float32), torch.zeros((0,), dtype=torch.bool)

    pad = (-T) % chunk
    total = T + pad

    feats = []
    mask = []
    D = None

    for i in range(0, total, chunk):
        mid = min(i + chunk // 2, T - 1)
        frame = vr[mid].asnumpy()

        crop, ok = _detect_face_crop(frame, size=224)

        if ok:
            emb = emotieff_embed(crop, device=device)
            if D is None:
                D = int(emb.shape[0])
            feats.append(torch.from_numpy(emb).to(torch.float32))
            mask.append(True)
        else:
            mask.append(False)
            if D is None:
                D = 1280
            feats.append(torch.zeros((D,), dtype=torch.float32))

    face = torch.stack(feats, dim=0)
    fmask = torch.tensor(mask, dtype=torch.bool)
    return face, fmask



_SKEL_PROJ = None

def _skel_projector(in_dim: int, out_dim: int = 512) -> nn.Module:
    global _SKEL_PROJ
    if _SKEL_PROJ is not None and getattr(_SKEL_PROJ, "_in_dim", None) == in_dim:
        return _SKEL_PROJ
    torch.manual_seed(42)
    proj = nn.Sequential(
        nn.Linear(in_dim, out_dim),
        nn.LayerNorm(out_dim),
        nn.GELU(),
    )
    proj._in_dim = in_dim
    _SKEL_PROJ = proj.eval()
    return _SKEL_PROJ


def _load_skeleton_csv(path: str) -> np.ndarray:
    df = pd.read_csv(path)
    arr = df.values.astype(np.float32)
    return arr  # (T,D)


@torch.no_grad()
def extract_skeleton_windows(skel_csv_path: str, device: str = "cpu", chunk: int = 32) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Return:
      skel:  (W,512) float32
      smask: (W,) bool
    """
    arr = _load_skeleton_csv(skel_csv_path)
    T, D = arr.shape
    if T <= 0:
        return torch.zeros((0, 512), dtype=torch.float32), torch.zeros((0,), dtype=torch.bool)

    pad = (-T) % chunk
    total = T + pad
    if pad:
        arr = np.concatenate([arr, np.repeat(arr[-1:], pad, axis=0)], axis=0)

    proj = _skel_projector(D, 512)

    feats = []
    mask = []
    for i in range(0, total, chunk):
        w = arr[i:i+chunk]
        valid = not np.allclose(w, 0)
        x = torch.from_numpy(w.mean(axis=0)).to(torch.float32)
        z = proj(x).detach().cpu()
        feats.append(z)
        mask.append(valid)

    skel = torch.stack(feats, dim=0).to(torch.float32)
    smask = torch.tensor(mask, dtype=torch.bool)
    return skel, smask




@torch.no_grad()
def build_and_cache_one(
    video_id: int,
    split: str,
    phase: int,
    feat_dir: str,
    chunk: int = 32,
    vpath: Optional[str] = None,
    spath: Optional[str] = None,
    do_ctx: bool = True,
    do_face: bool = True,
    do_skel: bool = True,
    device: str = "cuda",
):
    ps = cache_paths(feat_dir, video_id, split, phase)


    if vpath is None:
        if phase == 1:
            vpath = resolve_video_path_phase1(video_id, split)
        else:
            vpath = resolve_video_path_phase2(video_id)

    if spath is None:
        if phase == 1:
            spath = resolve_skeleton_path_phase1(video_id, split)
        else:
            spath = resolve_skeleton_path_phase2(video_id)

    if do_ctx or do_face:
        if vpath is None or (not os.path.exists(vpath)):
            raise FileNotFoundError(f"RGB video not found for id={video_id} split={split} phase={phase}")

    if do_skel:
        if spath is None or (not os.path.exists(spath)):
            raise FileNotFoundError(f"Skeleton csv not found for id={video_id} split={split} phase={phase}")

    if do_ctx and (not os.path.exists(ps["ctx"])):
        ctx = extract_ctx_stream(vpath, device=device, chunk=chunk)   # (W,512)
        torch.save(ctx, ps["ctx"])

    if do_face and (not os.path.exists(ps["face"])) and (not os.path.exists(ps["fmask"])):
        face, fmask = extract_face_windows(vpath, device=device, chunk=chunk)  # (W,D)
        torch.save(face, ps["face"])
        torch.save(fmask, ps["fmask"])

    if do_skel and (not os.path.exists(ps["skel"])) and (not os.path.exists(ps["smask"])):
        skel, smask = extract_skeleton_windows(spath, device="cpu", chunk=chunk)  # (W,512)
        torch.save(skel, ps["skel"])
        torch.save(smask, ps["smask"])

    return ps


Overwriting /content/src/feature_extract.py


In [ ]:
%%writefile /content/src/cache_features_cli.py
import argparse
import pandas as pd
from tqdm import tqdm
import torch
import traceback

from src.feature_extract import build_and_cache_one

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--phase", type=int, required=True)
    ap.add_argument("--feat_dir", required=True)
    ap.add_argument("--chunk", type=int, default=32)
    ap.add_argument("--start", type=int, default=0)
    ap.add_argument("--end", type=int, default=999999)
    ap.add_argument("--split_col", type=str, default="split")
    ap.add_argument("--device", choices=["auto","cpu","cuda"], default="auto")
    ap.add_argument("--do_ctx", action="store_true")
    ap.add_argument("--do_face", action="store_true")
    ap.add_argument("--do_skel", action="store_true")
    args = ap.parse_args()

    if args.device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = args.device

    if not (args.do_ctx or args.do_face or args.do_skel):
        args.do_ctx = args.do_face = args.do_skel = True

    print("DEVICE:", device)
    print("FEAT_DIR:", args.feat_dir)
    print("MODES: ctx=", args.do_ctx, "face=", args.do_face, "skel=", args.do_skel)

    df = pd.read_csv(args.csv)
    df = df.iloc[args.start:args.end].reset_index(drop=True)

    ok, bad = 0, 0

    for i, r in tqdm(df.iterrows(), total=len(df)):
        try:
            vid = int(r["video_id"])
            split = r.get(args.split_col, "test")

            build_and_cache_one(
                video_id=vid,
                split=split,
                phase=args.phase,
                feat_dir=args.feat_dir,
                chunk=args.chunk,
                vpath=None,
                spath=r.get("skeleton_path", None),
                do_ctx=args.do_ctx,
                do_face=args.do_face,
                do_skel=args.do_skel,
                device=device,
            )
            ok += 1

        except Exception:
            bad += 1
            print(" ERROR")
            print("row:", i, "video_id:", r.get("video_id"), "split:", r.get(args.split_col, ""))
            if "skeleton_path" in r:
                print("skeleton_path:", r.get("skeleton_path"))
            traceback.print_exc()

    print(f"\nDONE shard [{args.start}:{args.end}) ok={ok} bad={bad}")

if __name__ == "__main__":
    main()


Overwriting /content/src/cache_features_cli.py


In [ ]:
%%bash
CHUNK=64
SHARD=10
DEVICE=cuda
CSV=/content/phase1_all_with_split.csv
FEAT_DIR=/content/drive/MyDrive/miga_features_cache_agcn

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== CTX shard $START:$END (chunk=$CHUNK) ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 1 --feat_dir $FEAT_DIR \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_ctx

  START=$END
done


=== CTX shard 0:10 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= True face= False skel= False

DONE shard [0:10) ok=10 bad=0
=== CTX shard 10:20 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= True face= False skel= False

DONE shard [10:20) ok=10 bad=0
=== CTX shard 20:30 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= True face= False skel= False

DONE shard [20:30) ok=10 bad=0
=== CTX shard 30:40 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= True face= False skel= False

DONE shard [30:40) ok=10 bad=0
=== CTX shard 40:50 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= True face= False skel= False

DONE shard [40:50) ok=10 bad=0
=== CTX shard 50:60 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
M

100%|██████████| 5/5 [04:49<00:00, 57.96s/it]


In [ ]:
%%bash
CHUNK=32
SHARD=5
DEVICE=cuda
CSV=/content/phase1_all_with_split.csv
FEAT_DIR=/content/drive/MyDrive/miga_features_cache_agcn

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== FACE shard $START:$END (device=$DEVICE) ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 1 --feat_dir $FEAT_DIR \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_face

  START=$END
done


=== FACE shard 0:5 (device=cuda) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= True skel= False
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

DONE shard [0:5) ok=5 bad=0
=== FACE shard 5:10 (device=cuda) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= True skel= False

DONE shard [5:10) ok=5 bad=0
=== FACE shard 10:15 (device=cuda) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= True skel= False

DONE shard [10:15) ok=5 bad=0
=== FACE shard 15:20 (device=cuda) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= True skel= Fals

  0%|          | 0/5 [00:00<?, ?it/s]
  0%|          | 0.00/6.25M [00:00<?, ?B/s]
100%|██████████| 6.25M/6.25M [00:00<00:00, 53.4MB/s]
E0000 00:00:1768473915.285799   39579 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768473915.292454   39579 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768473915.312160   39579 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768473915.312196   39579 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768473915.312198   39579 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target mor

In [ ]:
%%bash
CHUNK=32
SHARD=10
DEVICE=cpu
CSV=/content/phase1_all_with_split.csv
FEAT_DIR=/content/drive/MyDrive/miga_features_cache_agcn

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== SKEL shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 1 --feat_dir $FEAT_DIR \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_skel

  START=$END
done


=== SKEL shard 0:10 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= False skel= True

DONE shard [0:10) ok=10 bad=0
=== SKEL shard 10:20 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= False skel= True

DONE shard [10:20) ok=10 bad=0
=== SKEL shard 20:30 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= False skel= True

DONE shard [20:30) ok=10 bad=0
=== SKEL shard 30:40 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= False skel= True

DONE shard [30:40) ok=10 bad=0
=== SKEL shard 40:50 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= False skel= True

DONE shard [40:50) ok=10 bad=0
=== SKEL shard 50:60 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn
MODES: ctx= False face= False skel= True

DONE shard [50:60) ok=10 

 60%|██████    | 6/10 [00:00<00:00,  7.22it/s]Traceback (most recent call last):
  File "/content/src/cache_features_cli.py", line 46, in main
    build_and_cache_one(
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 120, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/src/feature_extract.py", line 442, in build_and_cache_one
    if spath is None or (not os.path.exists(spath)):
                             ^^^^^^^^^^^^^^^^^^^^^
  File "<frozen genericpath>", line 19, in exists
TypeError: stat: path should be string, bytes, os.PathLike or integer, not float
 20%|██        | 2/10 [00:00<00:02,  2.89it/s]Traceback (most recent call last):
  File "/content/src/cache_features_cli.py", line 46, in main
    build_and_cache_one(
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 120, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "

In [ ]:
import torch, os
p = "/content/drive/MyDrive/miga_features_cache/be6f549156a8_face.pt"
x = torch.load(p, map_location="cpu")
print("file:", os.path.basename(p))
print("shape:", x.shape)
print("D_FACE =", x.shape[1])


file: be6f549156a8_face.pt
shape: torch.Size([201, 1280])
D_FACE = 1280


In [ ]:
%%writefile /content/src/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.GELU(),
            nn.Linear(d // 2, 1)
        )

    def forward(self, x, mask):
        score = self.scorer(x).squeeze(-1)
        score = score.float()

        score = score.masked_fill(~mask, -1e4)

        w = torch.softmax(score, dim=1).unsqueeze(-1)
        w = w.to(dtype=x.dtype)
        return (x * w).sum(dim=1)



class TriStreamModel(nn.Module):
    def __init__(
        self,
        d_ctx_in=512,
        d_face_in=1280,
        d_skel_in=512,
        d=512,
        n_layers=4,
        n_heads=4,
        dropout=0.3,
    ):
        super().__init__()


        self.ctx_in  = nn.Sequential(nn.Linear(d_ctx_in, d), nn.Dropout(dropout))
        self.face_in = nn.Sequential(nn.Linear(d_face_in, d), nn.Dropout(dropout))
        self.skel_in = nn.Sequential(nn.Linear(d_skel_in, d), nn.Dropout(dropout))

        enc = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=4 * d,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.ctx_enc  = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.face_enc = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.skel_enc = nn.TransformerEncoder(enc, num_layers=n_layers)

        self.ctx_pool  = AttnPool(d)
        self.face_pool = AttnPool(d)
        self.skel_pool = AttnPool(d)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(3 * d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1)
        )

    def forward(self, ctx, face, skel, time_mask, face_mask, skel_mask):

        pad_ignore = ~time_mask

        x_ctx  = self.ctx_in(ctx)
        x_face = self.face_in(face)
        x_skel = self.skel_in(skel)

        x_ctx  = self.ctx_enc(x_ctx,   src_key_padding_mask=pad_ignore)
        x_face = self.face_enc(x_face, src_key_padding_mask=pad_ignore)
        x_skel = self.skel_enc(x_skel, src_key_padding_mask=pad_ignore)

        ctx_vec  = self.ctx_pool(x_ctx,  time_mask)
        face_vec = self.face_pool(x_face, time_mask & face_mask)
        skel_vec = self.skel_pool(x_skel, time_mask & skel_mask)

        z = torch.cat([ctx_vec, face_vec, skel_vec], dim=-1)
        return self.head(z).squeeze(-1)


Overwriting /content/src/model.py


In [ ]:
%%writefile /content/src/ds.py
import os
import torch
from torch.utils.data import Dataset

from .feature_extract import cache_paths


_CTX_PROJ = None

def _ctx_to_512(x: torch.Tensor) -> torch.Tensor:
    """
    x: (T, D) on CPU
    returns (T, 512) on CPU
    """
    global _CTX_PROJ
    if x.ndim != 2:
        raise ValueError(f"ctx must be (T,D), got {tuple(x.shape)}")

    T, D = x.shape
    if D == 512:
        return x
    if D == 768:

        if _CTX_PROJ is None:
            _CTX_PROJ = torch.nn.Linear(768, 512, bias=False)
            torch.nn.init.orthogonal_(_CTX_PROJ.weight)
            _CTX_PROJ.eval()
        with torch.no_grad():
            return _CTX_PROJ(x.float()).to(dtype=torch.float32)
    raise ValueError(f"Unexpected ctx dim: {D} (expected 512 or 768)")

def _ensure_float32(x: torch.Tensor) -> torch.Tensor:
    return x.float() if x.dtype != torch.float32 else x


class Track3CachedDataset(Dataset):
    def __init__(self, df, feat_dir: str, phase: int = 1, has_label: bool = True):
        self.df = df.reset_index(drop=True)
        self.feat_dir = feat_dir
        self.phase = phase
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vid = int(row["video_id"])
        split = row.get("split", "test")

        ps = cache_paths(self.feat_dir, vid, split, self.phase)

        ctx  = torch.load(ps["ctx"],  map_location="cpu")
        face = torch.load(ps["face"], map_location="cpu")
        skel = torch.load(ps["skel"], map_location="cpu")

        fmask = torch.load(ps["fmask"], map_location="cpu")
        smask = torch.load(ps["smask"], map_location="cpu")

        ctx  = _ensure_float32(ctx)
        face = _ensure_float32(face)
        skel = _ensure_float32(skel)

        ctx = _ctx_to_512(ctx)

        T = min(ctx.shape[0], face.shape[0], skel.shape[0])
        ctx, face, skel = ctx[:T], face[:T], skel[:T]
        fmask, smask = fmask[:T].bool(), smask[:T].bool()

        out = {
            "id": vid,
            "ctx": ctx,
            "face": face,
            "skel": skel,
            "time_mask": torch.ones((T,), dtype=torch.bool),
            "face_mask": fmask,
            "skel_mask": smask,
        }
        if self.has_label:
            out["y"] = torch.tensor(float(row["label"]), dtype=torch.float32)
        return out



def pad_seq(list_TD, pad_value=0.0):
    T = max(x.shape[0] for x in list_TD)
    D = list_TD[0].shape[1]
    out = torch.full((len(list_TD), T, D), pad_value, dtype=torch.float32)
    tm  = torch.zeros((len(list_TD), T), dtype=torch.bool)
    for i, x in enumerate(list_TD):
        out[i, :x.shape[0]] = x.float()
        tm[i, :x.shape[0]] = True
    return out, tm


def collate_fn(batch):
    ctx, time_mask = pad_seq([b["ctx"] for b in batch])
    face, _        = pad_seq([b["face"] for b in batch])
    skel, _        = pad_seq([b["skel"] for b in batch])

    face_mask = torch.zeros_like(time_mask)
    skel_mask = torch.zeros_like(time_mask)

    for i, b in enumerate(batch):
        face_mask[i, :len(b["face_mask"])] = b["face_mask"]
        skel_mask[i, :len(b["skel_mask"])] = b["skel_mask"]

    out = {
        "id": [b["id"] for b in batch],
        "ctx": ctx,
        "face": face,
        "skel": skel,
        "time_mask": time_mask,
        "face_mask": face_mask,
        "skel_mask": skel_mask,
    }
    if "y" in batch[0]:
        out["y"] = torch.stack([b["y"] for b in batch])
    return out


Overwriting /content/src/ds.py


In [ ]:
import os, pandas as pd
from src.feature_extract import cache_paths

ROOT = "/content/miga_data"
FEAT_DIR = "/content/drive/MyDrive/miga_features_cache_agcn"

train_csv = f"{ROOT}/imigue_rgb_phase1/train_label.csv"
val_csv   = f"{ROOT}/imigue_rgb_phase1/validation_label.csv"

train_df = pd.read_csv(train_csv); train_df["split"]="train"
val_df   = pd.read_csv(val_csv);   val_df["split"]="val"

need = ["ctx","face","fmask","skel","smask"]

def has_cache(r):
    ps = cache_paths(FEAT_DIR, int(r["video_id"]), r["split"], 1)
    return all(os.path.exists(ps[k]) for k in need)

train_ok = train_df[train_df.apply(has_cache, axis=1)].reset_index(drop=True)
val_ok   = val_df[val_df.apply(has_cache, axis=1)].reset_index(drop=True)

print("train_ok:", len(train_ok), "/", len(train_df))
print("val_ok:", len(val_ok), "/", len(val_df))
print("val counts:", val_ok["label"].value_counts().to_dict())


train_ok: 241 / 245
val_ok: 10 / 10
val counts: {1: 10}


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

from src.ds import Track3CachedDataset, collate_fn
from src.model import TriStreamModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

D_FACE = 1280

sample_ds = Track3CachedDataset(train_ok.iloc[:1], feat_dir=FEAT_DIR, phase=1, has_label=True)
sample = sample_ds[0]
d_skel_in = sample["skel"].shape[1]
print("d_skel_in:", d_skel_in)

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none", pos_weight=self.pos_weight)
        p = torch.sigmoid(logits)
        pt = torch.where(targets == 1, p, 1 - p).clamp(1e-6, 1-1e-6)
        return (((1 - pt) ** self.gamma) * bce).mean()

def eval_auc(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for b in loader:
            logit = model(
                b["ctx"].to(device, non_blocking=True),
                b["face"].to(device, non_blocking=True),
                b["skel"].to(device, non_blocking=True),
                b["time_mask"].to(device, non_blocking=True),
                b["face_mask"].to(device, non_blocking=True),
                b["skel_mask"].to(device, non_blocking=True),
            )
            prob = torch.sigmoid(logit).detach().cpu().numpy()
            ys.append(b["y"].numpy())
            ps.append(prob)
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    if len(np.unique(ys)) < 2:
        return float("nan")
    return roc_auc_score(ys, ps)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y = train_ok["label"].values

best_auc = -1
best_state = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_ok, y), 1):
    tr_df = train_ok.iloc[tr_idx].reset_index(drop=True)
    va_df = train_ok.iloc[va_idx].reset_index(drop=True)

    tr_ds = Track3CachedDataset(tr_df, feat_dir=FEAT_DIR, phase=1, has_label=True)
    va_ds = Track3CachedDataset(va_df, feat_dir=FEAT_DIR, phase=1, has_label=True)

    tr_loader = DataLoader(tr_ds, batch_size=4, shuffle=True, num_workers=0, collate_fn=collate_fn, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=4, shuffle=False, num_workers=0, collate_fn=collate_fn, pin_memory=True)

    pos = (tr_df["label"] == 1).sum()
    neg = (tr_df["label"] == 0).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32).to(device)

    model = TriStreamModel(d_ctx_in=512, d_face_in=D_FACE, d_skel_in=d_skel_in,
                          d=512, n_layers=4, n_heads=4, dropout=0.3).to(device)

    crit = FocalLoss(gamma=1.0, pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

    scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))
    GRAD_ACCUM = 2
    patience = 4
    bad = 0
    fold_best = -1

    for epoch in range(1, 21):
        model.train()
        opt.zero_grad(set_to_none=True)
        losses = []

        for step, b in enumerate(tr_loader, 1):
            with torch.cuda.amp.autocast(enabled=(device=="cuda")):
                logit = model(
                    b["ctx"].to(device, non_blocking=True),
                    b["face"].to(device, non_blocking=True),
                    b["skel"].to(device, non_blocking=True),
                    b["time_mask"].to(device, non_blocking=True),
                    b["face_mask"].to(device, non_blocking=True),
                    b["skel_mask"].to(device, non_blocking=True),
                )
                loss = crit(logit, b["y"].to(device, non_blocking=True)) / GRAD_ACCUM

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)

            losses.append(loss.item() * GRAD_ACCUM)

        auc = eval_auc(model, va_loader)
        print(f"fold {fold} epoch {epoch}: loss={np.mean(losses):.4f} val_auc={auc:.4f}")

        if np.isfinite(auc) and auc > fold_best + 1e-4:
            fold_best = auc
            bad = 0
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

print("Best CV AUC:", best_auc)

if best_state is not None:
    out = "/content/drive/MyDrive/best_tristream_cv_agcn.pt"
    torch.save(best_state, out)
    print("Saved:", out)


device: cuda
d_skel_in: 512


/tmp/ipython-input-3531534122.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))
/tmp/ipython-input-3531534122.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


fold 1 epoch 1: loss=0.2065 val_auc=0.5789


/tmp/ipython-input-3531534122.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


fold 1 epoch 2: loss=0.2100 val_auc=0.3014


...

fold 5 epoch 6: loss=0.1523 val_auc=0.6462


/tmp/ipython-input-3531534122.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


fold 5 epoch 7: loss=0.1556 val_auc=0.6486


/tmp/ipython-input-3531534122.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda")):


fold 5 epoch 8: loss=0.1609 val_auc=0.6609
Best CV AUC: 0.6947368421052631
Saved: /content/drive/MyDrive/best_tristream_cv_agcn.pt


In [ ]:
ROOT = "/content/miga_data"
FEAT_DIR_P1 = "/content/drive/MyDrive/miga_features_cache_agcn"
FEAT_DIR_P2 = "/content/drive/MyDrive/miga_features_cache_agcn_p2"
CKPT_PATH = "/content/drive/MyDrive/best_tristream_cv.pt"


In [ ]:
import os, glob
import pandas as pd

ROOT = "/content/miga_data"

RGB_PHASE2 = f"{ROOT}/imigue_rgb_phase2"
SK_P2_TEST = f"{ROOT}/imigue_data_phase2/imigue_skeleton_test"
def vid4(x): return f"{int(x):04d}"

def resolve_video_path_phase2(video_id):
    v = vid4(video_id)
    p = os.path.join(RGB_PHASE2, v, f"{v}.mp4")
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(RGB_PHASE2, "**", f"{v}.mp4"), recursive=True)
    return hits[0] if hits else None

def resolve_skeleton_path_phase2(video_id, prefer_hand=True):
    v = vid4(video_id)
    p_hand  = os.path.join(SK_P2_TEST, v, f"{v}_light_hand.csv")
    p_light = os.path.join(SK_P2_TEST, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None

ids = sorted([int(os.path.basename(p)) for p in glob.glob(os.path.join(SK_P2_TEST, "[0-9][0-9][0-9][0-9]"))])

rows = []
miss_v, miss_s = 0, 0
for vid in ids:
    vpath = resolve_video_path_phase2(vid)
    spath = resolve_skeleton_path_phase2(vid)
    if vpath is None: miss_v += 1
    if spath is None: miss_s += 1
    rows.append({
        "video_id": vid,
        "split": "test",
        "video_path": vpath,
        "skeleton_path": spath
    })

phase2_df = pd.DataFrame(rows).sort_values("video_id").reset_index(drop=True)
print("Phase2 rows:", len(phase2_df), "missing video:", miss_v, "missing skeleton:", miss_s)

PHASE2_ALL = "/content/phase2_all_with_paths.csv"
phase2_df.to_csv(PHASE2_ALL, index=False)
print("Saved:", PHASE2_ALL)
phase2_df.head()


Phase2 rows: 104 missing video: 0 missing skeleton: 0
Saved: /content/phase2_all_with_paths.csv


,video_id,split,video_path,skeleton_path
0,54,test,/content/miga_data/imigue_rgb_phase2/0054/0054...,/content/miga_data/imigue_data_phase2/imigue_s...
1,67,test,/content/miga_data/imigue_rgb_phase2/0067/0067...,/content/miga_data/imigue_data_phase2/imigue_s...
2,69,test,/content/miga_data/imigue_rgb_phase2/0069/0069...,/content/miga_data/imigue_data_phase2/imigue_s...
3,70,test,/content/miga_data/imigue_rgb_phase2/0070/0070...,/content/miga_data/imigue_data_phase2/imigue_s...
4,71,test,/content/miga_data/imigue_rgb_phase2/0071/0071...,/content/miga_data/imigue_data_phase2/imigue_s...


In [ ]:
%%bash
CHUNK=32
SHARD=10
DEVICE=cuda

TOTAL=$(python - << 'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase2_all_with_paths.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== CTX shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv /content/phase2_all_with_paths.csv \
    --phase 2 \
    --feat_dir /content/drive/MyDrive/miga_features_cache_agcn_p2 \
    --chunk $CHUNK \
    --start $START --end $END \
    --device $DEVICE \
    --do_ctx

  START=$END
done


=== CTX shard 0:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= True face= False skel= False

DONE shard [0:10) ok=10 bad=0
=== CTX shard 10:20 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= True face= False skel= False

DONE shard [10:20) ok=10 bad=0
=== CTX shard 20:30 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= True face= False skel= False

DONE shard [20:30) ok=10 bad=0
=== CTX shard 30:40 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= True face= False skel= False

DONE shard [30:40) ok=10 bad=0
=== CTX shard 40:50 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= True face= False skel= False

DONE shard [40:50) ok=10 bad=0
=== CTX shard 50:60 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= True face= False skel= False

DONE sh

100%|██████████| 4/4 [00:00<00:00, 1696.04it/s]


In [ ]:
%%bash
CHUNK=32
SHARD=25
DEVICE=cuda

TOTAL=$(python - << 'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase2_all_with_paths.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== FACE shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv /content/phase2_all_with_paths.csv \
    --phase 2 \
    --feat_dir /content/drive/MyDrive/miga_features_cache_agcn_p2 \
    --chunk $CHUNK \
    --start $START --end $END \
    --device $DEVICE \
    --do_face

  START=$END
done


=== FACE shard 0:25 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= True skel= False

DONE shard [0:25) ok=25 bad=0
=== FACE shard 25:50 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= True skel= False

DONE shard [25:50) ok=25 bad=0
=== FACE shard 50:75 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= True skel= False

DONE shard [50:75) ok=25 bad=0
=== FACE shard 75:100 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= True skel= False

DONE shard [75:100) ok=25 bad=0
=== FACE shard 100:104 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= True skel= False

DONE shard [100:104) ok=4 bad=0


100%|██████████| 4/4 [00:00<00:00, 1633.14it/s]


In [ ]:
%%bash
CHUNK=32
SHARD=25
DEVICE=cuda

TOTAL=$(python - << 'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase2_all_with_paths.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== SKEL shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv /content/phase2_all_with_paths.csv \
    --phase 2 \
    --feat_dir /content/drive/MyDrive/miga_features_cache_agcn_p2 \
    --chunk $CHUNK \
    --start $START --end $END \
    --device $DEVICE \
    --do_skel

  START=$END
done


=== SKEL shard 0:25 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= False skel= True

DONE shard [0:25) ok=25 bad=0
=== SKEL shard 25:50 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= False skel= True

DONE shard [25:50) ok=25 bad=0
=== SKEL shard 50:75 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= False skel= True

DONE shard [50:75) ok=25 bad=0
=== SKEL shard 75:100 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= False skel= True

DONE shard [75:100) ok=25 bad=0
=== SKEL shard 100:104 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2
MODES: ctx= False face= False skel= True

DONE shard [100:104) ok=4 bad=0


100%|██████████| 4/4 [00:00<00:00, 1599.81it/s]


In [ ]:
import torch
from torch.utils.data import DataLoader
import pandas as pd

from src.ds import Track3CachedDataset, collate_fn

DEVICE = "cuda"

FEAT_DIR_P2 = "/content/drive/MyDrive/miga_features_cache_agcn_p2"
PHASE2_ALL = "/content/phase2_all_with_paths.csv"

phase2_df = pd.read_csv(PHASE2_ALL)

print("Columns:", phase2_df.columns.tolist())

test_ds = Track3CachedDataset(
    df=phase2_df,
    feat_dir=FEAT_DIR_P2,
    phase=2
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

print("Phase2 samples:", len(test_ds))


Columns: ['video_id', 'split', 'video_path', 'skeleton_path']
Phase2 samples: 104


In [ ]:
from src.model import TriStreamModel

MODEL_PATH = "/content/drive/MyDrive/best_tristream_cv_agcn.pt"

model = TriStreamModel(
    d_ctx_in=512,
    d_face_in=1280,
    d_skel_in=512,
    d=512,
    n_layers=4,
    n_heads=4,
    dropout=0.3
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

model.eval()

print("Model loaded successfully.")


Model loaded successfully.


In [ ]:
from src.ds import Track3CachedDataset, collate_fn
from torch.utils.data import DataLoader

FEAT_DIR_P2 = "/content/drive/MyDrive/miga_features_cache_agcn_p2"

test_ds = Track3CachedDataset(
    df=phase2_df,
    feat_dir=FEAT_DIR_P2,
    phase=2,
    has_label=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=True
)

print("Test samples:", len(test_ds))


Test samples: 104


In [ ]:
import numpy as np
from tqdm import tqdm

all_ids = []
all_probs = []

with torch.no_grad():
    for b in tqdm(test_loader):
        logit = model(
            b["ctx"].to(device, non_blocking=True),
            b["face"].to(device, non_blocking=True),
            b["skel"].to(device, non_blocking=True),
            b["time_mask"].to(device, non_blocking=True),
            b["face_mask"].to(device, non_blocking=True),
            b["skel_mask"].to(device, non_blocking=True),
        )

        print(f"logit shape: {logit.shape}")

        prob = torch.sigmoid(logit).cpu().numpy().flatten()

        video_ids = np.array(b["id"])

        all_probs.append(prob)
        all_ids.append(video_ids)

all_probs = np.concatenate(all_probs)
all_ids   = np.concatenate(all_ids)

print("Predictions:", all_probs.shape)
print("IDs:", all_ids.shape)


 15%|█▌        | 4/26 [00:00<00:01, 14.77it/s]

logit shape: torch.Size([4])
logit shape: torch.Size([4])
logit shape: torch.Size([4])
logit shape: torch.Size([4])


 23%|██▎       | 6/26 [00:00<00:01, 14.28it/s]

logit shape: torch.Size([4])
logit shape: torch.Size([4])
logit shape: torch.Size([4])


 38%|███▊      | 10/26 [00:00<00:01, 14.48it/s]

...

logit shape: torch.Size([4])
logit shape: torch.Size([4])
logit shape: torch.Size([4])


 92%|█████████▏| 24/26 [00:01<00:00, 13.05it/s]

logit shape: torch.Size([4])
logit shape: torch.Size([4])
logit shape: torch.Size([4])


100%|██████████| 26/26 [00:01<00:00, 13.80it/s]

logit shape: torch.Size([4])
Predictions: (104,)
IDs: (104,)


In [ ]:
submission = pd.DataFrame({
    "video_id": all_ids.astype(int),
    "label": all_probs.astype(float)
}).sort_values("video_id")

submission_path = "/content/drive/MyDrive/submission_phase2.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
submission.head()


Submission saved to: /content/drive/MyDrive/submission_phase2.csv


,video_id,label
0,54,0.669441
1,67,0.670642
2,69,0.668160
3,70,0.669676
4,71,0.669501
